# 動態爬蟲案例：MangaZ（教師版）

使用 Selenium 開啟免費閱讀頁面，並將漫畫頁面逐頁截圖。請依網站使用條款與著作權規範使用。

In [ ]:
from pathlib import Path
from selenium import webdriver
from selenium.common.exceptions import TimeoutException
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver.support.ui import WebDriverWait

options = Options()
options.add_experimental_option("excludeSwitches", ["enable-automation"])
options.add_experimental_option("useAutomationExtension", False)
options.add_argument("--disable-blink-features=AutomationControlled")
options.add_argument("--start-maximized")
options.add_argument("user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/121.0.0.0 Safari/537.36")

driver = webdriver.Chrome(options=options)
wait = WebDriverWait(driver, 15)
driver.get("https://www.mangaz.com/book/detail/157901")
print("Page Title:", driver.title)

In [ ]:
# 點擊「免費閱讀」並等待新視窗開啟
original_window = driver.current_window_handle
open_button = wait.until(
    EC.element_to_be_clickable((By.CSS_SELECTOR, "button.open-viewer.book-begin.ga"))
)
open_button.click()
wait.until(EC.number_of_windows_to_be(2))
new_window = next(handle for handle in driver.window_handles if handle != original_window)
driver.switch_to.window(new_window)

In [ ]:
# 點擊閱讀按鈕；以網址與常見按鈕樣式定位，避免依賴特定語言文字
read_now = wait.until(EC.element_to_be_clickable((
    By.CSS_SELECTOR,
    "a[href*='viewer'], a[href*='read'], .read-now a, a.btn"
)))
read_now.click()

In [ ]:
output_dir = Path("manga_pages")
output_dir.mkdir(exist_ok=True)
captured_sources = set()
page_number = 1

while True:
    images = wait.until(
        EC.presence_of_all_elements_located((By.CSS_SELECTOR, "div.page_image img.image"))
    )

    new_image_found = False
    for image in images:
        source = image.get_attribute("src")
        if image.is_displayed() and source and source not in captured_sources:
            file_path = output_dir / f"manga_page_{page_number:03d}.png"
            image.screenshot(str(file_path))
            captured_sources.add(source)
            print(f"已儲存：{file_path}")
            page_number += 1
            new_image_found = True

    try:
        next_page = WebDriverWait(driver, 5).until(
            EC.element_to_be_clickable((By.CSS_SELECTOR, "div.flip.flip-left"))
        )
        previous_sources = {img.get_attribute("src") for img in images}
        next_page.click()
        WebDriverWait(driver, 10).until(
            lambda d: any(
                img.get_attribute("src") not in previous_sources
                for img in d.find_elements(By.CSS_SELECTOR, "div.page_image img.image")
            )
        )
    except TimeoutException:
        print(f"擷取完成，共儲存 {len(captured_sources)} 張圖片。")
        break

In [ ]:
driver.quit()